In [ ]:
!pip -q install -U transformers accelerate sentencepiece huggingface_hub


In [ ]:
import re
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B-it"

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, switch to a T4 GPU runtime before running this notebook.")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print(f"Loaded {MODEL_ID}")
print(f"CUDA device: {torch.cuda.get_device_name(0)}")


In [ ]:
def clean_final_answer(text):

    text = re.sub(r"<turn\|>|<eos>|<bos>", "", text)
    text = re.sub(r"<\|/?[^>]+\|>|<[^>]+>", "", text)
    return text.strip()


def split_gemma_response(raw_text):

    patterns = [
        r"<\|channel\>thought\n(?P<thought>.*?)<channel\|>(?P<answer>.*?)(?:<turn\|>|<eos>|$)",
        r"<start_of_turn>thought\n(?P<thought>.*?)<end_of_turn>(?P<answer>.*?)(?:<end_of_turn>|<eos>|$)",
        r"<think>(?P<thought>.*?)</think>(?P<answer>.*?)(?:<eos>|$)",
    ]

    for pattern in patterns:
        match = re.search(pattern, raw_text, flags=re.DOTALL)
        if match:
            return match.group("thought").strip(), clean_final_answer(match.group("answer")), None

    parsed = None

    try:
        parsed = processor.parse_response(raw_text)
    except Exception:
        parsed = None

    if isinstance(parsed, dict):
        reasoning = parsed.get("thought") or parsed.get("thinking") or parsed.get("reasoning") or ""
        final_answer = parsed.get("answer") or parsed.get("final") or parsed.get("response") or ""
        if reasoning or final_answer:
            return str(reasoning).strip(), clean_final_answer(str(final_answer)), parsed

    if isinstance(parsed, (list, tuple)) and len(parsed) >= 2:
        return str(parsed[0]).strip(), clean_final_answer(str(parsed[1])), parsed

    return "", clean_final_answer(raw_text), parsed


@torch.inference_mode()
def ask_gemma(question, max_new_tokens=4096, do_sample=False, temperature=1.0, top_p=0.95, top_k=64):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": question},
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )

    inputs = processor(text=prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    if do_sample:
        generation_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)

    outputs = model.generate(**generation_kwargs)
    generated_ids = outputs[0][input_len:].tolist()
    raw_generation = processor.decode(generated_ids, skip_special_tokens=False)
    cot, final_answer, parsed = split_gemma_response(raw_generation)
    raw_output_tokens = raw_generation

    print("Input Question:")
    print()
    print(question)
    print()
    print("CoT:")
    print()
    print(cot if cot else "[No CoT block could be extracted from the raw generation]")
    print()
    print("Final Answer:")
    print()
    print(final_answer)
    print()
    print("============")
    print("Raw output tokens:")
    print()
    print(raw_output_tokens)

    return {
        "question": question,
        "cot": cot,
        "final_answer": final_answer,
        "raw_generation": raw_generation,
        "generated_token_ids": generated_ids,
        "raw_output_tokens": raw_output_tokens,
        "parsed": parsed,
    }


In [ ]:
question = input("Input Question: ").strip()
result = ask_gemma(question)
